# Representational Similarity Analysis (RSA) - Wilson et al. Replication

This notebook performs complete RSA analysis on preprocessed EEG data.

## Analysis Pipeline
1. Load preprocessed epochs
2. Define theoretical model RDMs
3. Compute neural RDMs from EEG data
4. Perform RSA correlation analysis
5. Statistical testing (permutation tests)
6. Temporal dynamics analysis
7. Visualization and reporting

## References
- Wilson et al. (original study)
- Kriegeskorte et al. (2008) - RSA methodology

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('white')
plt.rcParams['figure.figsize'] = (12, 6)

# Configuration
PROCESSED_DIR = Path('../data/processed')
RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print('Environment setup complete')

## 1. Load Preprocessed Data

Load the epochs generated from the preprocessing pipeline.

In [ ]:
# Load preprocessed epochs
# TODO: Update with actual path
# epochs = mne.read_epochs(PROCESSED_DIR / 'preprocessed_epochs-epo.fif')

# For demonstration, create sample epochs
info = mne.create_info(
    ch_names=['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2'],
    sfreq=250,
    ch_types='eeg'
)

n_epochs = 80
n_channels = 10
n_times = 300  # 1.2 seconds at 250 Hz

data = np.random.randn(n_epochs, n_channels, n_times)
events = np.array([[i * 1000, 0, (i % 4) + 1] for i in range(n_epochs)])
event_id = {'condition_1': 1, 'condition_2': 2, 'condition_3': 3, 'condition_4': 4}

epochs = mne.EpochsArray(data, info, events, tmin=-0.2, event_id=event_id)

print(f'Loaded {len(epochs)} epochs')
print(f'Conditions: {list(event_id.keys())}')
print(f'Time window: {epochs.times[0]:.3f} to {epochs.times[-1]:.3f} s')

## 2. Define Theoretical Model RDMs

Create representational dissimilarity matrices (RDMs) based on theoretical models.

In [ ]:
def create_rdm(dissimilarity_matrix, labels=None):
    """
    Create an RDM from a dissimilarity matrix.
    
    Parameters:
    -----------
    dissimilarity_matrix : ndarray
        Square dissimilarity matrix
    labels : list, optional
        Labels for conditions
    
    Returns:
    --------
    rdm : ndarray
        Representational dissimilarity matrix
    """
    rdm = np.array(dissimilarity_matrix)
    # Ensure symmetry
    rdm = (rdm + rdm.T) / 2
    # Zero diagonal
    np.fill_diagonal(rdm, 0)
    return rdm

print('RDM creation function defined')

In [ ]:
# TODO: Define Wilson et al. specific theoretical models

# Example: Semantic similarity model
semantic_rdm = create_rdm([
    [0, 1, 2, 3],
    [1, 0, 1, 2],
    [2, 1, 0, 1],
    [3, 2, 1, 0]
])

# Example: Categorical model (two categories)
categorical_rdm = create_rdm([
    [0, 0, 1, 1],
    [0, 0, 1, 1],
    [1, 1, 0, 0],
    [1, 1, 0, 0]
])

# Example: Perceptual similarity model
perceptual_rdm = create_rdm([
    [0, 1, 3, 2],
    [1, 0, 2, 3],
    [3, 2, 0, 1],
    [2, 3, 1, 0]
])

# Store models
model_rdms = {
    'Semantic': semantic_rdm,
    'Categorical': categorical_rdm,
    'Perceptual': perceptual_rdm
}

print(f'Defined {len(model_rdms)} theoretical model RDMs')

In [ ]:
# Visualize model RDMs
fig, axes = plt.subplots(1, len(model_rdms), figsize=(15, 4))

condition_labels = ['C1', 'C2', 'C3', 'C4']

for idx, (name, rdm) in enumerate(model_rdms.items()):
    im = axes[idx].imshow(rdm, cmap='RdYlBu_r', aspect='auto')
    axes[idx].set_title(f'{name} Model', fontsize=12, fontweight='bold')
    axes[idx].set_xticks(range(len(condition_labels)))
    axes[idx].set_yticks(range(len(condition_labels)))
    axes[idx].set_xticklabels(condition_labels)
    axes[idx].set_yticklabels(condition_labels)
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_rdms.png', dpi=300, bbox_inches='tight')
print('Model RDMs visualized')

## 3. Compute Neural RDMs

Calculate RDMs from EEG data at each time point.

In [ ]:
def compute_neural_rdm(epochs, time_idx=None, metric='correlation'):
    """
    Compute neural RDM from EEG epochs.
    
    Parameters:
    -----------
    epochs : mne.Epochs
        EEG epochs
    time_idx : int or slice, optional
        Time points to include. If None, average over all time.
    metric : str
        Distance metric ('correlation', 'euclidean', etc.)
    
    Returns:
    --------
    rdm : ndarray
        Neural RDM (conditions x conditions)
    """
    # Get average response for each condition
    conditions = sorted(epochs.event_id.keys())
    n_conditions = len(conditions)
    
    # Extract patterns for each condition
    patterns = []
    for cond in conditions:
        cond_epochs = epochs[cond].get_data()  # (n_epochs, n_channels, n_times)
        
        if time_idx is not None:
            cond_epochs = cond_epochs[:, :, time_idx]
        
        # Average over epochs and flatten
        pattern = cond_epochs.mean(axis=0).flatten()
        patterns.append(pattern)
    
    patterns = np.array(patterns)
    
    # Compute pairwise distances
    rdm = squareform(pdist(patterns, metric=metric))
    
    return rdm

print('Neural RDM computation function defined')

In [ ]:
# Compute neural RDM averaged over time window of interest
# Example: 300-500 ms window
time_window = (0.3, 0.5)
time_mask = (epochs.times >= time_window[0]) & (epochs.times <= time_window[1])

neural_rdm = compute_neural_rdm(epochs, time_idx=time_mask)

print(f'Computed neural RDM for time window {time_window[0]}-{time_window[1]} s')
print(f'RDM shape: {neural_rdm.shape}')

In [ ]:
# Visualize neural RDM
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

im = ax.imshow(neural_rdm, cmap='RdYlBu_r', aspect='auto')
ax.set_title(f'Neural RDM\n({time_window[0]}-{time_window[1]} s)', fontsize=12, fontweight='bold')
ax.set_xticks(range(len(condition_labels)))
ax.set_yticks(range(len(condition_labels)))
ax.set_xticklabels(condition_labels)
ax.set_yticklabels(condition_labels)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Dissimilarity')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'neural_rdm.png', dpi=300, bbox_inches='tight')

## 4. RSA Correlation Analysis

Correlate neural RDM with theoretical model RDMs.

In [ ]:
def compare_rdms(rdm1, rdm2, method='spearman'):
    """
    Correlate two RDMs.
    
    Parameters:
    -----------
    rdm1, rdm2 : ndarray
        RDMs to compare
    method : str
        Correlation method ('spearman' or 'pearson')
    
    Returns:
    --------
    correlation : float
        Correlation coefficient
    p_value : float
        P-value
    """
    # Extract upper triangle (excluding diagonal)
    triu_idx = np.triu_indices_from(rdm1, k=1)
    vec1 = rdm1[triu_idx]
    vec2 = rdm2[triu_idx]
    
    # Compute correlation
    if method == 'spearman':
        corr, p = spearmanr(vec1, vec2)
    else:
        corr, p = pearsonr(vec1, vec2)
    
    return corr, p

print('RDM comparison function defined')

In [ ]:
# Compare neural RDM with each model
rsa_results = {}

print('RSA Correlation Results:')
print('-' * 50)

for model_name, model_rdm in model_rdms.items():
    corr, p = compare_rdms(neural_rdm, model_rdm)
    rsa_results[model_name] = {'correlation': corr, 'p_value': p}
    
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'{model_name:15s}: r = {corr:6.3f}, p = {p:.4f} {sig}')

# Convert to DataFrame
rsa_df = pd.DataFrame(rsa_results).T
rsa_df.to_csv(RESULTS_DIR / 'rsa_correlations.csv')

In [ ]:
# Visualize RSA results
fig, ax = plt.subplots(figsize=(8, 5))

models = list(rsa_results.keys())
correlations = [rsa_results[m]['correlation'] for m in models]
p_values = [rsa_results[m]['p_value'] for m in models]

colors = ['green' if p < 0.05 else 'gray' for p in p_values]

bars = ax.bar(models, correlations, color=colors, alpha=0.7, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_ylabel('Correlation (Spearman r)', fontsize=12)
ax.set_xlabel('Theoretical Model', fontsize=12)
ax.set_title(f'RSA Results ({time_window[0]}-{time_window[1]} s)', fontsize=14, fontweight='bold')
ax.set_ylim(-1, 1)

# Add significance stars
for i, (bar, p) in enumerate(zip(bars, p_values)):
    height = bar.get_height()
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.05,
            sig, ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rsa_results.png', dpi=300, bbox_inches='tight')

## 5. Statistical Testing - Permutation Tests

Assess significance using permutation testing.

In [ ]:
def permutation_test(neural_rdm, model_rdm, n_permutations=1000):
    """
    Permutation test for RSA significance.
    
    Parameters:
    -----------
    neural_rdm : ndarray
        Neural RDM
    model_rdm : ndarray
        Model RDM
    n_permutations : int
        Number of permutations
    
    Returns:
    --------
    observed_corr : float
        Observed correlation
    p_value : float
        Permutation p-value
    null_distribution : ndarray
        Distribution of correlations under null hypothesis
    """
    # Observed correlation
    observed_corr, _ = compare_rdms(neural_rdm, model_rdm)
    
    # Permutation test
    null_corrs = []
    n_conditions = neural_rdm.shape[0]
    
    for _ in range(n_permutations):
        # Permute condition labels
        perm_idx = np.random.permutation(n_conditions)
        perm_rdm = neural_rdm[perm_idx, :][:, perm_idx]
        
        # Compute correlation with permuted RDM
        perm_corr, _ = compare_rdms(perm_rdm, model_rdm)
        null_corrs.append(perm_corr)
    
    null_corrs = np.array(null_corrs)
    
    # Calculate p-value
    p_value = np.sum(null_corrs >= observed_corr) / n_permutations
    
    return observed_corr, p_value, null_corrs

print('Permutation test function defined')

In [ ]:
# Run permutation tests
n_perms = 1000
perm_results = {}

print(f'Running permutation tests ({n_perms} permutations)...')
print('-' * 50)

for model_name, model_rdm in model_rdms.items():
    obs_corr, p_perm, null_dist = permutation_test(neural_rdm, model_rdm, n_perms)
    perm_results[model_name] = {
        'observed_correlation': obs_corr,
        'permutation_p': p_perm,
        'null_distribution': null_dist
    }
    
    sig = '***' if p_perm < 0.001 else '**' if p_perm < 0.01 else '*' if p_perm < 0.05 else 'ns'
    print(f'{model_name:15s}: r = {obs_corr:6.3f}, p_perm = {p_perm:.4f} {sig}')

In [ ]:
# Visualize null distributions
fig, axes = plt.subplots(1, len(perm_results), figsize=(15, 4))

for idx, (model_name, result) in enumerate(perm_results.items()):
    ax = axes[idx]
    
    # Plot null distribution
    ax.hist(result['null_distribution'], bins=30, color='gray', alpha=0.7, edgecolor='black')
    
    # Plot observed correlation
    ax.axvline(result['observed_correlation'], color='red', linewidth=2, 
               label=f'Observed (p={result["permutation_p"]:.3f})')
    
    ax.set_xlabel('Correlation')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{model_name} Model', fontweight='bold')
    ax.legend()

fig.suptitle('Permutation Test Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'permutation_tests.png', dpi=300, bbox_inches='tight')

## 6. Temporal Dynamics Analysis

Examine how RSA correlations evolve over time.

In [ ]:
# Define time windows for sliding analysis
window_size = 50  # 50 ms window
step_size = 10    # 10 ms step

sfreq = epochs.info['sfreq']
window_samples = int(window_size * sfreq / 1000)
step_samples = int(step_size * sfreq / 1000)

# Time points to analyze
n_times = len(epochs.times)
time_points = []
temporal_correlations = {model: [] for model in model_rdms.keys()}

print(f'Computing temporal RSA (window={window_size}ms, step={step_size}ms)...')

for start_idx in range(0, n_times - window_samples, step_samples):
    end_idx = start_idx + window_samples
    time_idx = slice(start_idx, end_idx)
    
    # Compute neural RDM for this time window
    neural_rdm_t = compute_neural_rdm(epochs, time_idx=time_idx)
    
    # Compare with each model
    for model_name, model_rdm in model_rdms.items():
        corr, _ = compare_rdms(neural_rdm_t, model_rdm)
        temporal_correlations[model_name].append(corr)
    
    # Store time point (center of window)
    center_time = epochs.times[start_idx + window_samples // 2]
    time_points.append(center_time)

time_points = np.array(time_points)
print(f'Computed RSA for {len(time_points)} time windows')

In [ ]:
# Plot temporal dynamics
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for (model_name, corrs), color in zip(temporal_correlations.items(), colors):
    ax.plot(time_points, corrs, label=model_name, linewidth=2, color=color)

ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Correlation with Model RDM', fontsize=12)
ax.set_title('Temporal Dynamics of RSA', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'temporal_rsa.png', dpi=300, bbox_inches='tight')

# Save temporal data
temporal_df = pd.DataFrame(temporal_correlations, index=time_points)
temporal_df.index.name = 'time_s'
temporal_df.to_csv(RESULTS_DIR / 'temporal_rsa.csv')

## 7. Wilson et al. Specific Analysis

### TODO: Implement Wilson et al. specific analyses

- Specific time windows of interest
- ROI (region of interest) analysis
- Cross-subject analysis
- Comparison with original results

In [ ]:
# TODO: Wilson et al. specific analysis code

# Example: Analyze specific time window from original study
# wilson_time_window = (0.250, 0.400)  # Example

# Example: ROI analysis (e.g., posterior electrodes)
# roi_channels = ['P3', 'P4', 'O1', 'O2']
# epochs_roi = epochs.copy().pick_channels(roi_channels)

print('TODO: Implement Wilson et al. specific analyses')

## 8. Summary and Export Results

Generate comprehensive summary of RSA results.

In [ ]:
# Create summary report
summary_report = []
summary_report.append('=' * 60)
summary_report.append('RSA ANALYSIS SUMMARY')
summary_report.append('Wilson et al. EEG RSA Replication')
summary_report.append('=' * 60)
summary_report.append('')
summary_report.append('ANALYSIS PARAMETERS:')
summary_report.append(f'  Time window: {time_window[0]:.3f} - {time_window[1]:.3f} s')
summary_report.append(f'  Number of epochs: {len(epochs)}')
summary_report.append(f'  Number of conditions: {len(event_id)}')
summary_report.append(f'  Number of models tested: {len(model_rdms)}')
summary_report.append('')
summary_report.append('RSA CORRELATION RESULTS:')

for model_name in model_rdms.keys():
    corr = rsa_results[model_name]['correlation']
    p = rsa_results[model_name]['p_value']
    p_perm = perm_results[model_name]['permutation_p']
    
    sig = '***' if p_perm < 0.001 else '**' if p_perm < 0.01 else '*' if p_perm < 0.05 else 'ns'
    
    summary_report.append(f'  {model_name}:')
    summary_report.append(f'    Correlation: {corr:.4f}')
    summary_report.append(f'    P-value (parametric): {p:.4f}')
    summary_report.append(f'    P-value (permutation): {p_perm:.4f} {sig}')
    summary_report.append('')

summary_report.append('=' * 60)

# Print summary
summary_text = '\n'.join(summary_report)
print(summary_text)

# Save summary
with open(RESULTS_DIR / 'rsa_summary.txt', 'w') as f:
    f.write(summary_text)

## Next Steps

1. ✓ RSA analysis complete
2. → Compare with Wilson et al. original findings
3. → Perform cross-subject analysis
4. → Create publication-ready figures
5. → Write up results section

## References

- Kriegeskorte, N., Mur, M., & Bandettini, P. (2008). Representational similarity analysis-connecting the branches of systems neuroscience. Frontiers in systems neuroscience, 2, 4.
- Nili, H., Wingfield, C., Walther, A., Su, L., Marslen-Wilson, W., & Kriegeskorte, N. (2014). A toolbox for representational similarity analysis. PLoS computational biology, 10(4), e1003553.